# Islamic AI — Step 1: Setup Colab Environment

**Before running this notebook:**
1. Go to `Runtime` → `Change runtime type`
2. Set Hardware accelerator to **T4 GPU**
3. Click Save
4. Then run all cells top to bottom

This notebook:
- Verifies your GPU is available
- Installs Unsloth (fast fine-tuning library)
- Mounts your Google Drive
- Creates the folder structure
- Runs a quick sanity check

**Estimated time:** 5–8 minutes

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if 'T4' in result.stdout or 'A100' in result.stdout or 'V100' in result.stdout:
    print('\n✅ GPU confirmed. Good to go!')
elif result.returncode != 0:
    print('\n❌ No GPU detected!')
    print('Go to Runtime → Change runtime type → T4 GPU → Save')
    raise SystemExit('Please enable GPU before continuing.')
else:
    print('\n⚠️  GPU detected but model unknown. Proceeding...')

## Cell 2 — Install Unsloth and Training Libraries

This takes 3–5 minutes on first run.

In [ ]:
%%capture
import sys

# Unsloth — 2x faster fine-tuning, 60% less VRAM
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

# Dataset and utility libraries
!pip install datasets jsonlines

print('Installation complete.')

In [ ]:
# Verify key imports
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('\n✅ All libraries imported successfully.')

## Cell 3 — Mount Google Drive

Your trained model will be saved to Google Drive so it is not lost when Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive mounted at /content/drive')

## Cell 4 — Create Folder Structure in Drive

In [ ]:
import os

BASE = '/content/drive/MyDrive/islamic_ai'

folders = [
    f'{BASE}/dataset',       # JSONL training files go here
    f'{BASE}/model/lora',    # Trained LoRA adapters saved here
    f'{BASE}/model/gguf',    # GGUF mobile model saved here
    f'{BASE}/logs',          # Training logs
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'  Created: {folder}')

print(f'\n✅ Folder structure ready at {BASE}')
print('\nNext: Upload your dataset files to:')
print(f'  {BASE}/dataset/')
print('Files needed: train.jsonl, eval.jsonl, test.jsonl')

## Cell 5 — Quick Model Load Test

This confirms Unsloth can load Llama 3.2 on your GPU. It downloads ~2GB — takes 2–3 minutes.

In [ ]:
from unsloth import FastLanguageModel

print('Loading Llama 3.2 3B Instruct (test load)...')
print('This downloads ~2GB on first run. Please wait...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = 2048,
    dtype = None,        # Auto-detect: bfloat16 on A100, float16 on T4
    load_in_4bit = True, # QLoRA: 4-bit quantization
)

print('\n✅ Model loaded successfully!')
print(f'Model parameters: {model.num_parameters() / 1e9:.2f}B')

# Free memory — we just tested the load
del model
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print('Memory freed. Ready for training notebook.')

## ✅ Setup Complete!

Everything is ready. Now do the following:

**Step 1:** Upload your dataset files to Google Drive:
- Open Google Drive → `islamic_ai` → `dataset`
- Upload `train.jsonl`, `eval.jsonl`, `test.jsonl`
- These files are in your project at: `02_dataset_formatting/formatted_output/`

**Step 2:** Open notebook `02_prepare_dataset.ipynb` and run it.

**Step 3:** Open notebook `03_train_model.ipynb` — this does the actual training.

---
*Note: Each Colab session gives you ~12 hours of free GPU time. The training takes approximately 8–10 hours on a T4 GPU.*